## Configurando o Ambiente

In [ ]:
%pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.7 MB/s eta 0:00:00


In [ ]:
from getpass import getpass
import os
os.environ['GROQ_API_KEY'] = getpass("Digite sua GROQ API Key: ")

Teste de funcionamento do Groq

In [ ]:
import os
from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of water to fishes",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Water is essential for the survival of fishes, and its importance cannot be overstated. Here are some reasons why water is crucial for fishes:

1. **Breathing and Respiration**: Fishes use gills to extract oxygen from the water, which is essential for respiration. Water is necessary for the exchange of oxygen and carbon dioxide, and without it, fishes would suffocate.
2. **Body Support and Buoyancy**: Water provides the necessary buoyancy and support for fishes to maintain their body shape and move around. The density of water helps fishes to conserve energy and move efficiently.
3. **Thermoregulation**: Water helps fishes to regulate their body temperature. Fishes are ectothermic, meaning they rely on external sources of heat to regulate their body temperature. Water helps to dissipate heat and maintain a stable temperature.
4. **Feeding and Digestion**: Water is essential for the digestion and absorption of food in fishes. Fishes use water to break down their food and absorb the nutr

## Criando o Agente

Iniciando a criação do **chat de memória** desse agente ReAct




In [ ]:
class Agent:
  def __init__(self, client, system):
      self.client = client
      self.system = system
      self.messages = []
      if self.system is not None:
          self.messages.append({"role": "system", "content": self.system})
  def __call__(self, message=""):
    if message:
      #salva msg do usuário
      self.messages.append({"role": "user", "content": message})
    result = self.execute()
    #salva msg da llm
    self.messages.append({"role": "assistant", "content": result})
    return result



  def execute(self):
    completion = self.client.chat.completions.create(
        messages = self.messages,
      model="llama-3.3-70b-versatile",
    )
    return completion.choices[0].message.content
#

Definição do system_prompt
Ele define como o modelo deve pensar e agir.

* Define o formato obrigatório da resposta
* Força o modelo a usar Thought / Action / PAUSE
* Instrui o modelo a não responder direto
* Ensina o padrão de interação






In [ ]:
system_prompt = """
You run in a loop of Thought, PAUSE, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this:

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

Definição das Ferramentas

In [ ]:
def  calculate(operation):
  return eval(operation)

def get_planet_mass(planet):
  match planet.lower():
    case "earth":
      return 5.972e24
    case "mars":
      return 0.642e24
    case "jupyter":
      return 1.898e27
    case "saturn":
      return 5.683e26
    case "uranus":
      return 8.681e25
    case "neptune":
      return 1.024e26
    case "mercury":
      return 3.285e23
    case "venus":
      return 4.867e24
    case _:
      return 0.0

## Inicializando nosso agente

In [ ]:
neil_tyson = Agent(client, system_prompt)


In [ ]:
result = neil_tyson("What is the mass of mars times 5?")

print(result)

Thought: To find the mass of Mars times 5, I first need to determine the mass of Mars. 

Action: get_planet_mass: Mars
PAUSE


### Explicação
Acima foi retornado o pensamento que a LLM gerou de acordo com o System Prompt;

Definiu a ação que seria necessária, chamando a ferramenta/função em questão com o parametro mencionado

Definiu um momento de pausa


---
Realizando o Teste da ferramenta que o Agente terá que usar





In [ ]:
observation = get_planet_mass("Mars")
print(observation)

6.42e+23


In [ ]:
next_prompt  = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: Now that I have the mass of Mars, which is 6.42e+23 kg, I need to multiply this by 5 to find the result.

Action: calculate: 6.42e+23 * 5
PAUSE


Olharemos o Histórico de mensagens

In [ ]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, PAUSE, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you h

Continuaremos com a sequencia de pensamento do agente

In [ ]:
observation = calculate("6.42e+23 * 5")
print(observation)

3.21e+24


In [ ]:
next_prompt  = f"Observation: {observation}"
result = neil_tyson(next_prompt)
print(result)

Thought: I have now calculated the mass of Mars times 5, which is 3.21e+24 kg. This is the final step in solving the problem.

Answer: The mass of Mars times 5 is 3.21e+24.


In [ ]:
neil_tyson.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, PAUSE, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you h



---

Criando um Loop do agente ReAct que definimos anteriormente

In [ ]:
import re
def agent_loop(max_iterations,system, query):
  agent = Agent(client, system_prompt)
  tools = ['calculate', 'get_planet_mass']
  next_prompt = query
  i=0
  while i < max_iterations:
    i += 1
    result = agent(next_prompt)
    print(result)

    if "PAUSE" in result and "Action" in result:
      action = re.findall(r'Action: ([a-z_]+): (.+)', result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1]

      if chosen_tool in tools:
        result_tool = eval(f"{chosen_tool}({'arg'})")
        next_prompt  =  f"Observation: {result_tool}"
      else:
        next_prompt = "Observation: Tools not found"

      print(next_prompt)
      continue
    if "Answer" in result:
        break

In [ ]:
agent_loop(max_iterations= 10, system = system_prompt, query = "What is the mass of mercury times 2 times the mass of earth?")

Thought: To find the mass of Mercury times 2 times the mass of Earth, I first need to find the mass of Mercury and the mass of Earth. I'll start by finding the mass of Mercury.

Action: get_planet_mass: Mercury
PAUSE
Observation: 3.285e+23
Thought: I now have the mass of Mercury, which is 3.285e+23 kg. Next, I need to find the mass of Earth. 

Action: get_planet_mass: Earth
PAUSE
Observation: 5.972e+24
Thought: I now have the mass of Earth, which is 5.972e+24 kg. I can now calculate the mass of Mercury times 2 and then multiply it by the mass of Earth.

Action: calculate: 2 * 3.285e+23
PAUSE
Observation: 6.57e+23
Thought: I now have the result of the mass of Mercury times 2, which is 6.57e+23 kg. Now, I need to multiply this result by the mass of Earth.

Action: calculate: 6.57e+23 * 5.972e+24
PAUSE
Observation: 3.923604e+48
Thought: I have now calculated the mass of Mercury times 2 times the mass of Earth, which is 3.923604e+48 kg. This is the final result.

Answer: The mass of Mercur